# Loss Functions: MSE, BCE, CrossEntropy

Each loss measures a different prediction task — regression vs binary vs multi-class classification.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. MSE for regression

In [ ]:
pred = torch.linspace(-2, 2, 100)
target = torch.zeros_like(pred)
mse = (pred - target) ** 2

## 2. BCE for binary probabilities

In [ ]:
p = torch.linspace(0.01, 0.99, 100)
y1 = torch.ones_like(p)
bce_pos = -torch.log(p)
bce_neg = -torch.log(1 - p)

## 3. Plot MSE and BCE curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(pred.numpy(), mse.numpy(), 'b-', lw=2)
axes[0].set_title('MSE = (pred - 0)²'); axes[0].set_xlabel('prediction')
axes[1].plot(p.numpy(), bce_pos.numpy(), label='y=1', lw=2)
axes[1].plot(p.numpy(), bce_neg.numpy(), label='y=0', lw=2)
axes[1].legend(); axes[1].set_title('Binary Cross-Entropy'); axes[1].set_xlabel('p(pred=1)')
plt.tight_layout(); plt.show()

## 4. Cross-entropy over class logits

In [ ]:
logits = torch.linspace(-3, 3, 100)
# CE when true class is 0 vs 1 (2-class softmax)
ce0 = -F.log_softmax(torch.stack([logits, torch.zeros_like(logits)]), dim=0)[0]
ce1 = -F.log_softmax(torch.stack([torch.zeros_like(logits), logits]), dim=0)[1]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(logits.numpy(), ce0.numpy(), label='true class 0', lw=2)
ax.plot(logits.numpy(), ce1.numpy(), label='true class 1', lw=2)
ax.set_xlabel('logit for favored class'); ax.set_ylabel('CE loss')
ax.legend(); ax.set_title('Cross-entropy (2-class)')
plt.tight_layout(); plt.show()

## 5. Compare loss values on synthetic batch

In [ ]:
gen = torch.randn(32, 4)
reg_t = torch.randn(32, 1)
bin_p = torch.sigmoid(torch.randn(32))
bin_y = torch.randint(0, 2, (32,)).float()
cls_logits = torch.randn(32, 3)
cls_y = torch.randint(0, 3, (32,))

losses = {
    'MSE': F.mse_loss(gen[:, :1], reg_t).item(),
    'BCE': F.binary_cross_entropy(bin_p, bin_y).item(),
    'CE': F.cross_entropy(cls_logits, cls_y).item(),
}
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(losses.keys(), losses.values(), color=['steelblue', 'coral', 'seagreen'], edgecolor='black')
ax.set_title('Sample batch loss values'); ax.set_ylabel('loss')
plt.tight_layout(); plt.show()